In [1]:
import os
import importlib
from dotenv import load_dotenv, find_dotenv

# Nạp biến môi trường từ file .env
load_dotenv(find_dotenv(usecwd=True), override=False)

# Ưu tiên biến đã có trong notebook, rồi đến .env
api_key = (
 os.getenv("GOOGLE_API_KEY")
)

if not api_key:
    raise ValueError("Không tìm thấy API key. Thêm GOOGLE_API_KEY vào file .env")

try:
    # SDK mới: google-genai
    import sys
    import subprocess
    import importlib

    # Update SDK to avoid `Model.__init__(..., thinking=...)` errors
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "google-genai"])
    importlib.invalidate_caches()
    from google import genai
    client = genai.Client(api_key=api_key)

    print("Các model khả dụng:")
    try:
        models = client.models.list()
    except Exception as e:
        err = str(e)
        if "API_KEY_INVALID" in err or "API key expired" in err:
            print("GOOGLE_API_KEY đã hết hạn/không hợp lệ. Hãy tạo key mới và cập nhật file .env.")
            models = []
        else:
            raise
    for model in models:
        print("-", model.name)

except ImportError:
    # SDK cũ: google-generativeai
    genai_old = importlib.import_module("google.generativeai")
    genai_old.configure(api_key=api_key)

    print("Các model khả dụng:")
    for model in genai_old.list_models():
        print("-", model.name)

Các model khả dụng:
- models/gemini-2.5-flash
- models/gemini-2.5-pro
- models/gemini-2.0-flash
- models/gemini-2.0-flash-001
- models/gemini-2.0-flash-lite-001
- models/gemini-2.0-flash-lite
- models/gemini-2.5-flash-preview-tts
- models/gemini-2.5-pro-preview-tts
- models/gemma-3-1b-it
- models/gemma-3-4b-it
- models/gemma-3-12b-it
- models/gemma-3-27b-it
- models/gemma-3n-e4b-it
- models/gemma-3n-e2b-it
- models/gemini-flash-latest
- models/gemini-flash-lite-latest
- models/gemini-pro-latest
- models/gemini-2.5-flash-lite
- models/gemini-2.5-flash-image
- models/gemini-2.5-flash-lite-preview-09-2025
- models/gemini-3-pro-preview
- models/gemini-3-flash-preview
- models/gemini-3.1-pro-preview
- models/gemini-3.1-pro-preview-customtools
- models/gemini-3.1-flash-lite-preview
- models/gemini-3-pro-image-preview
- models/nano-banana-pro-preview
- models/gemini-3.1-flash-image-preview
- models/gemini-robotics-er-1.5-preview
- models/gemini-2.5-computer-use-preview-10-2025
- models/deep-r

In [2]:
from io import BytesIO
from pathlib import Path
import shutil

import fitz  # PyMuPDF
import pytesseract
from PIL import Image
from pypdf import PdfReader, PdfWriter


def resolve_tesseract_cmd(tesseract_cmd: str | None = None) -> str:
    """Tìm đường dẫn tesseract.exe từ input, PATH hoặc vị trí cài mặc định trên Windows."""
    if tesseract_cmd:
        candidate = Path(tesseract_cmd)
        if not candidate.exists():
            raise FileNotFoundError(f"Không tìm thấy tesseract.exe tại: {candidate}")
        return str(candidate)

    in_path = shutil.which("tesseract")
    if in_path:
        return in_path

    common_windows_paths = [
        Path(r"C:\Program Files\Tesseract-OCR\tesseract.exe"),
        Path(r"C:\Program Files (x86)\Tesseract-OCR\tesseract.exe"),
    ]
    for candidate in common_windows_paths:
        if candidate.exists():
            return str(candidate)

    raise RuntimeError(
        "Không tìm thấy Tesseract. Hãy cài Tesseract OCR hoặc truyền tesseract_cmd."
    )


def ocr_single_pdf(
    input_pdf: Path,
    output_pdf: Path,
    lang: str = "vie+eng",
    dpi: int = 300,
    tesseract_cmd: str | None = None,
    verbose: bool = True,
) -> Path:
    """OCR 1 file PDF và tạo PDF mới có text layer để searchable."""
    resolved_tesseract = resolve_tesseract_cmd(tesseract_cmd)
    pytesseract.pytesseract.tesseract_cmd = resolved_tesseract

    input_pdf = Path(input_pdf)
    output_pdf = Path(output_pdf)

    if not input_pdf.exists():
        raise FileNotFoundError(f"Không tìm thấy file: {input_pdf}")

    output_pdf.parent.mkdir(parents=True, exist_ok=True)

    # Render từng trang PDF thành ảnh rồi OCR -> PDF page có text layer
    src_doc = fitz.open(str(input_pdf))
    writer = PdfWriter()
    zoom = dpi / 72.0
    matrix = fitz.Matrix(zoom, zoom)

    try:
        for page_index in range(len(src_doc)):
            page = src_doc.load_page(page_index)
            pix = page.get_pixmap(matrix=matrix, alpha=False)
            img = Image.open(BytesIO(pix.tobytes("png")))

            ocr_pdf_bytes = pytesseract.image_to_pdf_or_hocr(
                img,
                extension="pdf",
                lang=lang,
            )
            ocr_reader = PdfReader(BytesIO(ocr_pdf_bytes))
            writer.add_page(ocr_reader.pages[0])

        with output_pdf.open("wb") as f:
            writer.write(f)
    finally:
        src_doc.close()

    if verbose:
        print(f"✅ OCR xong: {input_pdf.name} -> {output_pdf}")

    return output_pdf


def ocr_pdf_folder(
    input_dir: str | Path = "File_PDFs",
    output_dir: str | Path = "File_PDFs_OCR",
    pattern: str = "*.pdf",
    lang: str = "vie+eng",
    dpi: int = 300,
    tesseract_cmd: str | None = None,
) -> list[Path]:
    """OCR tất cả PDF trong thư mục input và lưu PDF mới sang thư mục output."""
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)

    if not input_dir.exists():
        raise FileNotFoundError(f"Không tìm thấy thư mục input: {input_dir}")

    resolved_tesseract = resolve_tesseract_cmd(tesseract_cmd)
    print(f"🔎 Tesseract: {resolved_tesseract}")

    pdf_files = sorted(input_dir.glob(pattern))
    if not pdf_files:
        print(f"⚠️ Không tìm thấy PDF nào trong: {input_dir}")
        return []

    output_dir.mkdir(parents=True, exist_ok=True)
    created_files: list[Path] = []

    print(f"📂 Input:  {input_dir}")
    print(f"📁 Output: {output_dir}")
    print(f"📄 Tổng số file: {len(pdf_files)}")

    for pdf_path in pdf_files:
        out_path = output_dir / f"{pdf_path.stem}_ocr.pdf"
        try:
            created = ocr_single_pdf(
                input_pdf=pdf_path,
                output_pdf=out_path,
                lang=lang,
                dpi=dpi,
                tesseract_cmd=resolved_tesseract,
                verbose=True,
            )
            created_files.append(created)
        except Exception as exc:
            print(f"❌ Lỗi OCR {pdf_path.name}: {exc}")

    print(f"\n🎉 Hoàn tất OCR. Tạo được {len(created_files)}/{len(pdf_files)} file.")
    return created_files


# Example chạy ngay:
generated_pdfs = ocr_pdf_folder(
    input_dir="File_PDFs",
    output_dir="File_PDFs_OCR",
    pattern="*.pdf",
    lang="vie+eng",
    dpi=300,
)

print("\nDanh sách file OCR đã tạo:")
for p in generated_pdfs:
    print("-", p)

🔎 Tesseract: C:\Program Files\Tesseract-OCR\tesseract.exe
📂 Input:  File_PDFs
📁 Output: File_PDFs_OCR
📄 Tổng số file: 2
✅ OCR xong: BanMOTa_CNTT_2020-2024.pdf -> File_PDFs_OCR\BanMOTa_CNTT_2020-2024_ocr.pdf
✅ OCR xong: CAMNANGTSVSGU2022.pdf -> File_PDFs_OCR\CAMNANGTSVSGU2022_ocr.pdf

🎉 Hoàn tất OCR. Tạo được 2/2 file.

Danh sách file OCR đã tạo:
- File_PDFs_OCR\BanMOTa_CNTT_2020-2024_ocr.pdf
- File_PDFs_OCR\CAMNANGTSVSGU2022_ocr.pdf
